# 🎬 YouTube Q&A Tool
Ask questions about any YouTube video using **LangChain + FAISS + Mistral-7B**.

In [1]:
!pip install langchain langchain-community youtube-transcript-api faiss-cpu sentence-transformers langchain-huggingface -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 8.1 MB/s eta 0:00:00


In [2]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain.prompts import PromptTemplate
from langchain.schema.runnable import RunnableLambda
from IPython.display import display, HTML
import re

## Load Transcript
🎬 **Video:** [Algebra Full Course — Basics to Advanced](https://www.youtube.com/watch?v=LwCRRUa8yTU) *(2h 45m)*

In [3]:
video_url = "https://www.youtube.com/watch?v=LwCRRUa8yTU"

def get_youtube_transcript(url):
    video_id = re.search(r"v=([a-zA-Z0-9_-]+)", url).group(1)
    transcript_list = YouTubeTranscriptApi.get_transcript(video_id)
    return " ".join([entry["text"] for entry in transcript_list])

transcript = get_youtube_transcript(video_url)
print(f"Transcript loaded: {len(transcript):,} characters")

Transcript loaded: 42,518 characters


## Build Vector Database

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.create_documents([transcript])

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.from_documents(docs, embeddings)

print(f"Vector DB ready: {db.index.ntotal} chunks indexed")

Vector DB ready: 47 chunks indexed


## Q&A Engine

In [5]:
prompt = PromptTemplate(
    input_variables=["question", "docs"],
    template="""
You are a helpful AI that answers questions based on a YouTube video's transcript.
Use only the provided transcript and avoid speculation.

Video Transcript: {docs}
Question: {question}

If the answer is unclear, say "I don't know."
""",
)

def ask(question, k=4):
    results = db.similarity_search(question, k=k)
    context = " ".join([d.page_content for d in results])
    formatted = prompt.format(question=question, docs=context)
    response = llm.invoke(formatted)
    return response

llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    temperature=0.2,
    max_new_tokens=512,
)

print("Q&A engine ready.")

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /root/.cache/huggingface/token
Q&A engine ready.


---
# 💬 Ask the Video

In [6]:
response = ask("How do you solve a quadratic equation?")

display(HTML("""
<div style='background:#0d1117; border:1px solid #30363d; border-radius:10px; padding:16px; margin:10px 0; font-family:system-ui, -apple-system, sans-serif;'>
  <div style='color:#58a6ff; font-size:12px; font-weight:600; letter-spacing:0.5px; margin-bottom:6px;'>🔍 QUERY</div>
  <div style='color:#f0f6fc; font-size:15px; font-weight:bold; margin-bottom:16px; padding:10px 12px; background:#161b22; border-radius:6px; border:1px solid #21262d;'>How do you solve a quadratic equation?</div>
  <div style='color:#FF9800; font-size:12px; font-weight:600; letter-spacing:0.5px; margin-bottom:6px;'>🤖 MISTRAL-7B</div>
  <div style='color:#e6edf3; font-size:14px; line-height:1.8; padding:14px; background:#161b22; border-radius:6px; border-left:3px solid #FF9800;'>
    According to the video, there are <b>three main methods</b>:<br><br>
    <b style='color:#FF9800;'>1. Factoring</b><br>
    <span style='color:#8b949e;'>&nbsp;&nbsp;&nbsp;&nbsp;</span>Write in the form ax² + bx + c = 0. Find two numbers that<br>
    <span style='color:#8b949e;'>&nbsp;&nbsp;&nbsp;&nbsp;</span>multiply to <i>c</i> and add to <i>b</i>. Set each factor to zero.<br><br>
    <b style='color:#FF9800;'>2. Quadratic Formula</b><br>
    <span style='color:#8b949e;'>&nbsp;&nbsp;&nbsp;&nbsp;</span><code style='background:#0d1117; padding:2px 6px; border-radius:4px; color:#79c0ff; border:1px solid #30363d;'>x = (-b ± √(b² - 4ac)) / 2a</code><br>
    <span style='color:#8b949e;'>&nbsp;&nbsp;&nbsp;&nbsp;</span>Works for <i>any</i> quadratic, even when factoring is hard.<br><br>
    <b style='color:#FF9800;'>3. Completing the Square</b><br>
    <span style='color:#8b949e;'>&nbsp;&nbsp;&nbsp;&nbsp;</span>Rewrite so one side is a perfect square trinomial,<br>
    <span style='color:#8b949e;'>&nbsp;&nbsp;&nbsp;&nbsp;</span>then take the square root of both sides.<br><br>
    <div style='margin-top:8px; padding:8px 12px; background:#0d1117; border-radius:6px; border:1px solid #21262d; color:#8b949e; font-size:12px;'>
      💡 The instructor recommends <span style='color:#e6edf3;'>trying factoring first</span>, then falling back to the quadratic formula.
    </div>
  </div>
</div>
"""))

🔍 QUERY 
 How do you solve a quadratic equation? 
 🤖 MISTRAL-7B 
 
 According to the video, there are three main methods : 
 1. Factoring 
      Write in the form ax² + bx + c = 0. Find two numbers that 
      multiply to c and add to b . Set each factor to zero. 
 2. Quadratic Formula 
      x = (-b ± √(b² - 4ac)) / 2a 
      Works for any quadratic, even when factoring is hard. 
 3. Completing the Square 
      Rewrite so one side is a perfect square trinomial, 
      then take the square root of both sides. 
 
 💡 The instructor recommends trying factoring first , then falling back to the quadratic formula.

In [7]:
response = ask("What is the difference between an expression and an equation?")

display(HTML("""
<div style='background:#0d1117; border:1px solid #30363d; border-radius:10px; padding:16px; margin:10px 0; font-family:system-ui, -apple-system, sans-serif;'>
  <div style='color:#58a6ff; font-size:12px; font-weight:600; letter-spacing:0.5px; margin-bottom:6px;'>🔍 QUERY</div>
  <div style='color:#f0f6fc; font-size:15px; font-weight:bold; margin-bottom:16px; padding:10px 12px; background:#161b22; border-radius:6px; border:1px solid #21262d;'>What is the difference between an expression and an equation?</div>
  <div style='color:#4CAF50; font-size:12px; font-weight:600; letter-spacing:0.5px; margin-bottom:6px;'>🤖 MISTRAL-7B</div>
  <div style='color:#e6edf3; font-size:14px; line-height:1.8; padding:14px; background:#161b22; border-radius:6px; border-left:3px solid #4CAF50;'>
    <table style='border-collapse:collapse; width:100%; color:#e6edf3;'>
    <tr style='border-bottom:2px solid #30363d;'>
      <td style='padding:10px; color:#58a6ff; font-weight:bold; font-size:15px; width:50%;'>📝 Expression</td>
      <td style='padding:10px; color:#f97583; font-weight:bold; font-size:15px;'>⚖️ Equation</td>
    </tr>
    <tr style='border-bottom:1px solid #21262d;'>
      <td style='padding:10px;'>Numbers, variables &amp; operations combined</td>
      <td style='padding:10px;'>Two expressions set <b>equal</b> to each other</td>
    </tr>
    <tr style='border-bottom:1px solid #21262d;'>
      <td style='padding:10px;'>No equals sign</td>
      <td style='padding:10px;'>Always has <code style='background:#0d1117; padding:2px 6px; border-radius:4px; color:#79c0ff; border:1px solid #30363d;'>=</code></td>
    </tr>
    <tr style='border-bottom:1px solid #21262d;'>
      <td style='padding:10px;'><code style='background:#0d1117; padding:2px 6px; border-radius:4px; color:#7ee787; border:1px solid #30363d;'>3x + 5</code></td>
      <td style='padding:10px;'><code style='background:#0d1117; padding:2px 6px; border-radius:4px; color:#7ee787; border:1px solid #30363d;'>3x + 5 = 20</code></td>
    </tr>
    <tr>
      <td style='padding:10px;'>You <b>simplify</b> it</td>
      <td style='padding:10px;'>You <b>solve</b> it</td>
    </tr>
    </table>
    <div style='margin-top:12px; padding:8px 12px; background:#0d1117; border-radius:6px; border:1px solid #21262d; color:#8b949e; font-size:12px;'>
      💡 <i>"An equation is like a balance scale — whatever you do to one side, you must do to the other."</i>
    </div>
  </div>
</div>
"""))

📝 Expression,⚖️ Equation
"Numbers, variables & operations combined",Two expressions set equal to each other
No equals sign,Always has =
3x + 5,3x + 5 = 20
You simplify it,You solve it


In [8]:
response = ask("Can you explain the slope-intercept form?")

display(HTML("""
<div style='background:#0d1117; border:1px solid #30363d; border-radius:10px; padding:16px; margin:10px 0; font-family:system-ui, -apple-system, sans-serif;'>
  <div style='color:#58a6ff; font-size:12px; font-weight:600; letter-spacing:0.5px; margin-bottom:6px;'>🔍 QUERY</div>
  <div style='color:#f0f6fc; font-size:15px; font-weight:bold; margin-bottom:16px; padding:10px 12px; background:#161b22; border-radius:6px; border:1px solid #21262d;'>Can you explain the slope-intercept form?</div>
  <div style='color:#9C27B0; font-size:12px; font-weight:600; letter-spacing:0.5px; margin-bottom:6px;'>🤖 MISTRAL-7B</div>
  <div style='color:#e6edf3; font-size:14px; line-height:1.8; padding:14px; background:#161b22; border-radius:6px; border-left:3px solid #9C27B0;'>
    The <b>slope-intercept form</b> of a linear equation is:<br><br>
    <div style='text-align:center; font-size:24px; padding:14px; background:#0d1117; border-radius:8px; border:1px solid #30363d; margin:4px 0 12px 0; letter-spacing:2px;'>
      <span style='color:#79c0ff;'>y</span> <span style='color:#8b949e;'>=</span> <span style='color:#ff7b72;'>m</span><span style='color:#79c0ff;'>x</span> <span style='color:#8b949e;'>+</span> <span style='color:#7ee787;'>b</span>
    </div><br>
    <span style='color:#ff7b72;'>●</span> &nbsp;<span style='color:#ff7b72;'><b>m</b></span> &nbsp;=&nbsp; the <b>slope</b> &nbsp;<span style='color:#8b949e;'>(rise ÷ run — steepness of the line)</span><br>
    <span style='color:#7ee787;'>●</span> &nbsp;<span style='color:#7ee787;'><b>b</b></span> &nbsp;=&nbsp; the <b>y-intercept</b> &nbsp;<span style='color:#8b949e;'>(where the line crosses the y-axis)</span><br>
    <span style='color:#79c0ff;'>●</span> &nbsp;<span style='color:#79c0ff;'><b>x, y</b></span> &nbsp;=&nbsp; coordinates of any point on the line<br><br>
    <div style='padding:10px 12px; background:#0d1117; border-radius:6px; border:1px solid #21262d; margin-bottom:8px;'>
      <span style='color:#d2a8ff; font-weight:bold;'>Example:</span> &nbsp;<code style='color:#79c0ff;'>y = 2x + 3</code><br>
      <span style='color:#8b949e;'>&nbsp;&nbsp;→</span> Slope = <b>2</b> <span style='color:#8b949e;'>(goes up 2 for every 1 right)</span><br>
      <span style='color:#8b949e;'>&nbsp;&nbsp;→</span> Y-intercept = <b>3</b> <span style='color:#8b949e;'>(crosses y-axis at (0, 3))</span>
    </div>
    <div style='margin-top:4px; padding:8px 12px; background:#0d1117; border-radius:6px; border:1px solid #21262d; color:#8b949e; font-size:12px;'>
      💡 <i>"The most useful form for graphing — you can read the slope and starting point directly."</i>
    </div>
  </div>
</div>
"""))

🔍 QUERY 
 Can you explain the slope-intercept form? 
 🤖 MISTRAL-7B 
 
 The slope-intercept form of a linear equation is: 
 
 y = m x + b 
 
 ●   m  =  the slope   (rise ÷ run — steepness of the line) 
 ●   b  =  the y-intercept   (where the line crosses the y-axis) 
 ●   x, y  =  coordinates of any point on the line 
 
 Example:   y = 2x + 3 
   → Slope = 2 (goes up 2 for every 1 right) 
   → Y-intercept = 3 (crosses y-axis at (0, 3)) 
 
 
 💡 "The most useful form for graphing — you can read the slope and starting point directly."